In [3]:
from scipy import stats

# Hospital type is coded as 1 = general/medical-surgical and 0 = specialized
TYPE_COL = 'Gen Med & Sug or Not'
EXPENSE_COL = 'Total Expense'
ADMISSION_COL = 'Admissions'


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the pre-cleaned dataset
df = pd.read_csv('../data/processed/aha_cleaned.csv')
print(f"Loaded {df.shape[0]} rows and {df.shape[1]} columns.")

Loaded 1300 rows and 21 columns.


In [4]:
# Map the numeric hospital-type flag to readable group names for these tests
hospital_type = df[TYPE_COL].map({1: 'General', 0: 'Specialized'})

# ANOVA: compare total expenses across hospital types
expense_groups = [
    df.loc[hospital_type == name, EXPENSE_COL].dropna()
    for name in ('General', 'Specialized')
]

anova_result = stats.f_oneway(*expense_groups)
print('ANOVA — Total Expense by Hospital Type')
print(f'F-statistic: {anova_result.statistic:.4f}')
print(f'p-value: {anova_result.pvalue:.6g}')

anova_alpha = 0.05
if anova_result.pvalue < anova_alpha:
    print('Result: Significant difference in mean expenses among hospital types.')
else:
    print('Result: No significant difference in mean expenses among hospital types.')

# Welch t-test: compare admissions between General and Specialized hospitals
general_admissions = df.loc[hospital_type == 'General', ADMISSION_COL].dropna()
specialized_admissions = df.loc[hospital_type == 'Specialized', ADMISSION_COL].dropna()
welch_ttest = stats.ttest_ind(general_admissions, specialized_admissions, equal_var=False)

print('\nWelch t-test — Admissions: General vs. Specialized')
print(f'T-statistic: {welch_ttest.statistic:.4f}')
print(f'Degrees of freedom: {welch_ttest.df:.2f}')
print(f'p-value: {welch_ttest.pvalue:.6g}')

if welch_ttest.pvalue < anova_alpha:
    print('Result: Significant difference in mean admissions between groups.')
else:
    print('Result: No significant difference in mean admissions between groups.')

print('\nAdmissions mean (General):', general_admissions.mean())
print('Admissions mean (Specialized):', specialized_admissions.mean())


ANOVA — Total Expense by Hospital Type
F-statistic: 61.4415
p-value: 9.44428e-15
Result: Significant difference in mean expenses among hospital types.

Welch t-test — Admissions: General vs. Specialized
T-statistic: 17.9328
Degrees of freedom: 1243.93
p-value: 3.86074e-64
Result: Significant difference in mean admissions between groups.

Admissions mean (General): 10170.397274633124
Admissions mean (Specialized): 2353.1965317919075
